# Lid-driven cavity: the paper's optimiser, on this problem

Steady incompressible Navier-Stokes at $\mathrm{Re}=100$ on $[0,1]^2$; the lid ($y=1$) moves
with $u=4x(1-x)$, the other walls are no-slip.

**No reimplementation.** DSGNAR is the reference implementation of the paper

> J. Webb, S. Jerad, C. Cartis, *An Optimisation Framework for the Well-Conditioned Training of
> Physics-Informed Neural Networks*, arXiv:2607.02194

vendored from `github.com/wephy/physics-informed-neural-networks` in
`Evolution_try/vendor/dsgnar_pinn`, driven step by step exactly as its own `trainer64` does it
(float32 pre-pass, then float64, 24 radius probes, window 3, target rho 0.075, accept iff
rho > 0).  SSBroyden is the merged Crunch (`Jax/Crunch/Optimizers/minimize_backtracking.py`)
minimising **the same objective** — $\sum_i w_i\,\mathrm{mean}(r_i^2)$ over the same point sets —
from **the same starting parameters**.

Both are graded every iteration against the numerical reference field
(`ldc/reference_gen/ldc_reference.npz`, Re=100, from `make_ldc_reference.py`), so the figure shows
the relative L2 error, not only the loss: a smaller loss does not imply a better solution.

Hub notes: this needs the CUDA kernel (`PINN (jax, CUDA)`); XLA's caches go to `/tmp`, and
autotuning is off (the shared HAMi slice cannot satisfy the autotuner's temporary allocations).

### Sizes

Cell 3 has a single switch: `MODE = "smoke" | "a30" | "paper"`.  The reference implementation
builds a **dense Jacobian per macro-batch**, so its memory is set by
`batch_size * 3 * n_params * 8` bytes (plus the sketch matrix `s^2` and XLA temporaries, ~2-3x).

| mode | hidden | d | points | sketch | batch | dense Jacobian | notes |
|---|---|---|---|---|---|---|---|
| `smoke` | 40-40-40 | 3.5k | 2^11 / 2^10 | 256 | 256 | 0.02 GiB | minutes, even on CPU |
| `a30` | 48-48-48 | 5.0k | 2^13 / 2^12 | 1665 (=d/3) | 1024 | 0.11 GiB | fits the shared 19.3 GiB slice with room |
| `paper` | 60-60-60-60 | 11.3k | 2^15 / 2^14 | 4000 | 4096 | 1.04 GiB | the paper's numbers; needs headroom |


In [1]:
import os, pathlib, sys, tempfile, time

def _repo_root(start=None):
    starts = [pathlib.Path(start or pathlib.Path.cwd()).resolve(),
              pathlib.Path.home() / "serafin" / "Julia" / "PINN",   # hub node (JupyterLab)
              pathlib.Path.home() / "Julia" / "PINN"]               # login node
    for base in starts:
        for p in (base, *base.parents):
            if (p / "Stationary" / "stationary" / "dsgnar.py").exists():
                return p
    raise RuntimeError("repo root not found")

REPO = _repo_root()
# the paper's OWN package, vendored; no reimplementation is used below
sys.path.insert(0, str(REPO / "Evolution_try" / "vendor" / "dsgnar_pinn"))
sys.path.insert(0, str(REPO / "Jax"))       # Crunch: SSBroyden (merged 2026-10-17)

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")   # HAMi slice: no autotune workspaces

CACHE_DIR = pathlib.Path(tempfile.gettempdir()) / f"jaxcache-{os.environ.get('USER', 'user')}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGDIR = REPO / "ldc" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import jax, jax.numpy as jnp
import equinox as eqx
import matplotlib.pyplot as plt
import pinn
from pinn.optimiser import Optimiser          # the paper's sketched trust-region LM
from Crunch.Optimizers.minimize_backtracking import minimize as minimize_qn

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_compilation_cache_dir", str(CACHE_DIR))

print(f"pid {os.getpid()} | repo {REPO}")
print(f"cache {CACHE_DIR}   backend {jax.default_backend()} | {[str(d) for d in jax.devices()]}")
if jax.default_backend() != "gpu" and pathlib.Path.home().joinpath("serafin").is_dir():
    raise RuntimeError("CPU kernel on the hub: pick 'PINN (jax, CUDA)' before running")

pid 9550 | repo /serafin/reula/Julia/PINN
cache /tmp/jaxcache-reula   backend gpu | ['cuda:0']


In [2]:
class LidDrivenCavityProblem(pinn.Problem):
    """Steady lid-driven cavity at Re = 100; outputs (u, v, p)."""

    x_min, x_max = 0.0, 1.0
    Re = 100.0
    problem_name = "LidDrivenCavity"
    ref_path = str(REPO / "ldc" / "reference_gen" / "ldc_reference.npz")

    def __init__(self, *, n_pde, n_bc, n_ic=0):
        self.n_pde, self.n_bc, self.n_ic = n_pde, n_bc, n_ic

    def residual_fns(self):
        return {"pde": self.pde_residual, "bc": self.bc_residual}

    def pde_residual(self, model, coords):
        u_ = lambda c: model(c)[0]
        v_ = lambda c: model(c)[1]
        p_ = lambda c: model(c)[2]
        u, v = u_(coords), v_(coords)
        inv_Re = 1.0 / self.Re
        res_u = (u * op.grad(u_, coords, 0) + v * op.grad(u_, coords, 1)
                 + op.grad(p_, coords, 0)
                 - inv_Re * (op.grad2(u_, coords, 0) + op.grad2(u_, coords, 1)))
        res_v = (u * op.grad(v_, coords, 0) + v * op.grad(v_, coords, 1)
                 + op.grad(p_, coords, 1)
                 - inv_Re * (op.grad2(v_, coords, 0) + op.grad2(v_, coords, 1)))
        res_div = op.grad(u_, coords, 0) + op.grad(v_, coords, 1)
        return jnp.array([res_u, res_v, 2.0 * res_div])

    def bc_residual(self, model, coords):
        u, v = model(coords)[0], model(coords)[1]
        u_target = jnp.where(jnp.isclose(coords[1], 1.0), 4.0 * coords[0] * (1.0 - coords[0]),
                             0.0)
        return jnp.array([u - u_target, v, 0.0])

    def samplers(self):
        return {"pde": self._sample_pde, "bc": self._sample_bc}

    def _sample_pde(self, key, n, *_):
        # 7/8 bulk, 1/16 near the moving lid, 1/16 in the top corners
        n_lid, n_corner = n // 16, n // 16
        n_bulk = n - n_lid - n_corner
        k_bulk, k_lx, k_ly, k_l, k_r = jax.random.split(key, 5)
        bulk = jax.random.uniform(k_bulk, (n_bulk, 2), minval=0.0, maxval=1.0)
        lid = jnp.stack([jax.random.uniform(k_lx, (n_lid,)),
                         jax.random.uniform(k_ly, (n_lid,), minval=0.9, maxval=1.0)], axis=1)
        half = n_corner // 2
        s1 = jax.random.uniform(k_l, (half, 2))
        left = jnp.stack([0.1 * s1[:, 0]**2, 1.0 - 0.1 * s1[:, 1]**2], axis=1)
        s2 = jax.random.uniform(k_r, (half, 2))
        right = jnp.stack([1.0 - 0.1 * s2[:, 0]**2, 1.0 - 0.1 * s2[:, 1]**2], axis=1)
        return jnp.concatenate([bulk, lid, left, right], axis=0)

    def _sample_bc(self, key, n, *_):
        n_corner = n // 16
        n_uniform = n - n_corner
        half = n_corner // 2
        k_pt, k_dim, k_face, k_l, k_r = jax.random.split(key, 5)
        pts = jax.random.uniform(k_pt, (n_uniform, 2), minval=0.0, maxval=1.0)
        face_dim = jax.random.randint(k_dim, (n_uniform,), 0, 2)
        face_val = jax.random.randint(k_face, (n_uniform,), 0, 2).astype(pts.dtype)
        pts = pts.at[jnp.arange(n_uniform), face_dim].set(face_val)
        left = jnp.stack([0.1 * jax.random.uniform(k_l, (half,)), jnp.ones((half,))], axis=1)
        right = jnp.stack([1.0 - 0.1 * jax.random.uniform(k_r, (half,)),
                           jnp.ones((half,))], axis=1)
        return jnp.concatenate([pts, left, right], axis=0)


op, sampling = pinn.operators, pinn.sampling

In [3]:
# ── size presets ─────────────────────────────────────────────────────────────────────
# The memory driver of the paper's optimiser is its DENSE Jacobian per macro-batch:
#     batch_size * 3 residual components * n_params * 8 bytes
# plus the sketch matrix (s^2) and its SVD.  The other buffers are small.
#   "smoke"  : a few minutes even on a CPU, for checking the pipeline.
#   "a30"    : fits the shared 19.3 GiB HAMi slice of the hub's A30 with a wide margin
#              (dense Jacobian ~0.12 GiB, sketch ~0.02 GiB); ~10-20 min for 200 steps.
#   "paper"  : the paper's own production numbers (2^15 points, s=4000, 60-60-60-60).
#              Its dense Jacobian alone is ~1.1 GiB and XLA needs several times that,
#              so use it only on a larger/whole GPU.
MODE = "a30"                       # "smoke" | "a30" | "paper"

PRESETS = {
    "smoke": dict(hidden=(40, 40, 40), n_pde=2 ** 11, n_bc=2 ** 10, sketch=256,
                  batch=2 ** 8, probe=2 ** 7, float32_steps=2, dsgnar_steps=40,
                  qn_maxiter=200, min_radius=1e-2),
    "a30":   dict(hidden=(48, 48, 48), n_pde=2 ** 13, n_bc=2 ** 12, sketch=1665,
                  batch=1024, probe=128, float32_steps=4, dsgnar_steps=400,
                  qn_maxiter=10000, min_radius=1e-14),
    "paper": dict(hidden=(60, 60, 60, 60), n_pde=2 ** 15, n_bc=2 ** 14, sketch=4000,
                  batch=2 ** 12, probe=2 ** 8, float32_steps=10, dsgnar_steps=200,
                  qn_maxiter=1000, min_radius=1e-10),
}
P = PRESETS[MODE]
HIDDEN, N_PDE, N_BC = P["hidden"], P["n_pde"], P["n_bc"]
SKETCH, BATCH, PROBE = P["sketch"], P["batch"], P["probe"]
FLOAT32_STEPS, DSGNAR_STEPS = P["float32_steps"], P["dsgnar_steps"]
QN_MAXITER, MIN_RADIUS = P["qn_maxiter"], P["min_radius"]
PDE_WEIGHT, TARGET_RHO = 1e-2, 0.075

# the paper code asserts these divisibilities (points per macro-batch, probe blocks)
assert N_PDE % BATCH == 0 and N_BC % BATCH == 0, (N_PDE, N_BC, BATCH)
assert N_PDE % PROBE == 0 and N_BC % PROBE == 0, (N_PDE, N_BC, PROBE)

def siren_params(in_dim, hidden, out_dim=3):
    n, prev = 0, in_dim
    for h in hidden:
        n += h * prev + h
        prev = h
    return n + out_dim * prev + out_dim

D_PARAMS = siren_params(2, HIDDEN)
JAC_GIB = BATCH * 3 * D_PARAMS * 8 / 2 ** 30
SKETCH_GIB = SKETCH ** 2 * 8 / 2 ** 30
try:
    mem = jax.devices()[0].memory_stats()
    limit = f"{mem['bytes_limit'] / 2**30:.1f} GiB"
except Exception:
    limit = "unknown"
print(f"MODE={MODE}: hidden={HIDDEN} d={D_PARAMS} | n_pde={N_PDE} n_bc={N_BC} | "
      f"sketch={SKETCH} batch={BATCH} probe={PROBE}")
print(f"  memory estimate: dense Jacobian {JAC_GIB:.2f} GiB + sketch matrix "
      f"{SKETCH_GIB:.3f} GiB (+ XLA temporaries, ~2-3x the Jacobian) | GPU {limit}")
print(f"  budget: DSGNAR {DSGNAR_STEPS} steps (stops when radius < {10 * MIN_RADIUS:.0e}), "
      f"SSBroyden {QN_MAXITER} iterations | min_radius {MIN_RADIUS:.0e}")

MODE=a30: hidden=(48, 48, 48) d=4995 | n_pde=8192 n_bc=4096 | sketch=1665 batch=1024 probe=128
  memory estimate: dense Jacobian 0.11 GiB + sketch matrix 0.021 GiB (+ XLA temporaries, ~2-3x the Jacobian) | GPU 13.5 GiB
  budget: DSGNAR 400 steps (stops when radius < 1e-13), SSBroyden 10000 iterations | min_radius 1e-14


In [4]:
key = jax.random.PRNGKey(0)
problem = LidDrivenCavityProblem(n_pde=N_PDE, n_bc=N_BC)
model = pinn.SIREN(key, LidDrivenCavityProblem, time_dependent=False, hidden_dims=HIDDEN,
                   periodic_bc=False, n_inputs=2, n_outputs=3)
params, static = eqx.partition(model, eqx.is_array)
flat0, unflatten = jax.flatten_util.ravel_pytree(params)

# the paper's conditions: one point set per residual, each with its weight (Algorithm 2)
fns, samplers = problem.residual_fns(), problem.samplers()
key, sk = jax.random.split(key)
CONDITIONS = tuple((fns[name], samplers[name](sk, n, 0.0, 0.0))
                   for name, n in (("pde", N_PDE), ("bc", N_BC)))
WEIGHTS = (PDE_WEIGHT, 1.0)

# grading: the paper's loss, and the relative L2 error against the reference field
def paper_loss(flat_or_params, conditions=CONDITIONS, weights=WEIGHTS):
    m = flat_or_params if isinstance(flat_or_params, eqx.Module) else \
        eqx.combine(unflatten(flat_or_params), static)
    total = 0.0
    for (fn, pts), w in zip(conditions, weights):
        r = jax.vmap(lambda c: fn(m, c))(pts)
        total = total + w * jnp.mean(r ** 2)
    return total
paper_loss_jit = jax.jit(paper_loss)
paper_loss_grad = jax.jit(jax.value_and_grad(paper_loss))

ref = pinn.Reference.load(problem.ref_path)
REF_COORDS = jnp.asarray(ref.coords)
REF_SOL = np.asarray(ref.solution)
CHANNELS = [str(c) for c in ref.channels]

@jax.jit
def _predict(m, coords):
    return jax.vmap(lambda c: m(c))(coords)

def flat_of(model_params):
    f, _ = jax.flatten_util.ravel_pytree(model_params)
    return f

def rel_l2(flat_or_params):
    m = flat_or_params if isinstance(flat_or_params, eqx.Module) else \
        eqx.combine(unflatten(flat_or_params), static)
    coords = REF_COORDS.astype(jnp.float64 if jax.config.jax_enable_x64 else jnp.float32)
    pred = np.asarray(_predict(m, coords))
    out = {}
    for i, ch in enumerate(CHANNELS):
        a, b = pred[:, i], REF_SOL[:, i]
        if ch == "p":
            a, b = a - a.mean(), b - b.mean()
        out[ch] = float(np.linalg.norm(a - b) / np.linalg.norm(b))
    out["mean"] = float(np.mean([out[c] for c in CHANNELS]))
    return out

d = int(flat0.size)
jac_gib = BATCH * 3 * d * 8 / 2 ** 30
try:
    limit = jax.devices()[0].memory_stats()["bytes_limit"] / 2 ** 30
    fit = f"~{3.0 * jac_gib + SKETCH**2 * 8 / 2**30:.2f} GiB of buffers on a {limit:.1f} GiB card"
except Exception:
    fit = f"~{3.0 * jac_gib + SKETCH**2 * 8 / 2**30:.2f} GiB of buffers"
print(f"{d} parameters | dense Jacobian {jac_gib:.2f} GiB -> {fit}")
print(f"reference: {ref.describe()}")
print("initial: paper loss %.4e | rel L2 %s" % (
    float(paper_loss_jit(flat0)), {k: f"{v:.3f}" for k, v in rel_l2(flat0).items()}))

[SIREN] raw 2D | ω₀=1.0 ω_h=1.0 | learnable ω | embed(2)→48→48→48→3 | 4,998 params
4998 parameters | dense Jacobian 0.11 GiB -> ~0.36 GiB of buffers on a 13.5 GiB card
reference: 16641 eval points in 2D, channels ['u', 'v', 'p'], plot x×y (129×129)
initial: paper loss 6.3471e-01 | rel L2 {'u': '6.256', 'v': '3.828', 'p': '4.003', 'mean': '4.696'}


In [5]:
# ── DSGNAR: the paper's own Optimiser, driven as its trainer64 does ───────────────────
#   pass 1: float32, `FLOAT32_STEPS` iterations (the paper's early-divergence guard)
#   pass 2: float64 until the trust region falls below 10 * MIN_RADIUS
# Accepted iff rho > 0 (paper's reference code); 24 probes on [D/3, 3D]; target rho 0.075.
solver = Optimiser(residual_sketch=SKETCH, parameter_sketch=SKETCH, n_hashes=4,
                   sub_batch_size=4, batch_size=BATCH, probe_batch_size=PROBE,
                   n_probes=24, window_scale=3.0, min_radius=MIN_RADIUS, max_radius=1e3,
                   pchip_grid=512, newton_iters=80)

ds_hist = []          # (iteration, paper loss, rel L2 mean, radius, rho, accepted, pass)
step_total = 0
key, k2 = jax.random.split(key)
opt_state = solver.init_state(k2, initial_radius=1.0)
t0 = time.perf_counter()

for bits, n_steps in ((32, FLOAT32_STEPS), (64, DSGNAR_STEPS)):
    jax.config.update("jax_enable_x64", bits == 64)          # the trainer's two passes
    dtype = jnp.float64 if bits == 64 else jnp.float32
    params = jax.tree_util.tree_map(
        lambda x: x.astype(dtype) if eqx.is_inexact_array(x) else x, params)
    # fresh point sets for this pass, in this pass's precision (as trainer64/_build_conditions)
    key, sk = jax.random.split(key)
    conds = tuple((fns[name], samplers[name](sk, n, 0.0, 0.0))
                  for name, n in (("pde", N_PDE), ("bc", N_BC)))
    for k in range(n_steps):
        loss, params, opt_state, metrics = solver.step(
            params, static, opt_state, conds, TARGET_RHO, WEIGHTS)
        step_total += 1
        accepted = bool(metrics["accepted"])
        rl = rel_l2(params)["mean"]
        ds_hist.append((step_total, float(loss), rl, float(opt_state.radius),
                        float(metrics["rho"]), accepted, bits))
        if step_total % 5 == 0 or k == 0 or not accepted:
            print(f"[dsgnar] {step_total:4d} (float{bits})  loss {float(loss):.4e}  "
                  f"relL2 {rl:.4f}  radius {float(opt_state.radius):.2e}  "
                  f"rho {float(metrics['rho']):+.3f}  {'acc' if accepted else 'rej'}")
        if float(opt_state.radius) < 10 * MIN_RADIUS:
            print(f"[dsgnar] radius below threshold after {step_total} iterations")
            break
    CONDITIONS_QN = conds          # the float64 pass's point sets: SSBroyden's objective
    if bits == 64:
        break
    else:
        continue
jax.config.update("jax_enable_x64", True)
ds_params, ds_flat = params, flat_of(params)
np.save(REPO / "ldc" / "dsgnar_flat.npy", np.asarray(ds_flat))
print(f"[dsgnar] done: {step_total} iterations in {time.perf_counter() - t0:.1f}s | "
      f"paper loss {float(paper_loss_jit(ds_flat)):.4e} | rel L2 {rel_l2(ds_flat)['mean']:.4f}")

# SSBroyden minimises the paper loss on the SAME point sets the float64 pass used
paper_loss_qn = jax.jit(lambda flat: paper_loss(flat, CONDITIONS_QN, WEIGHTS))
paper_loss_qn_grad = jax.jit(jax.value_and_grad(lambda flat: paper_loss(flat, CONDITIONS_QN, WEIGHTS)))

/home/reula/serafin/Julia/PINN/Stationary/.venv/lib/python3.13/site-packages/jax/_src/numpy/util.py:84: UserWarning: Explicitly requested dtype float64 requested in dtype is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  to_dtype, weak_type = dtypes.lattice_result_type(*args)


[dsgnar]    1 (float32)  loss 1.0695e-01  relL2 2.8692  radius 3.00e+00  rho +0.824  acc
[dsgnar]    5 (float64)  loss 7.7977e-02  relL2 1.5651  radius 2.99e+00  rho +0.080  acc
[dsgnar]   10 (float64)  loss 5.4017e-02  relL2 1.3008  radius 1.15e+00  rho +0.072  acc
[dsgnar]   15 (float64)  loss 3.6485e-02  relL2 1.3035  radius 1.64e+00  rho +0.080  acc
[dsgnar]   20 (float64)  loss 2.4619e-02  relL2 1.0233  radius 9.32e-01  rho +0.072  acc
[dsgnar]   25 (float64)  loss 1.6667e-02  relL2 1.0728  radius 1.43e+00  rho +0.078  acc
[dsgnar]   30 (float64)  loss 1.1520e-02  relL2 0.5028  radius 9.81e-01  rho +0.070  acc
[dsgnar]   35 (float64)  loss 7.8327e-03  relL2 0.5266  radius 8.47e-01  rho +0.067  acc
[dsgnar]   40 (float64)  loss 5.1858e-03  relL2 0.4198  radius 8.70e-01  rho +0.076  acc
[dsgnar]   45 (float64)  loss 3.4999e-03  relL2 0.2551  radius 5.19e-01  rho +0.070  acc
[dsgnar]   50 (float64)  loss 2.3396e-03  relL2 0.2014  radius 5.37e-01  rho +0.075  acc
[dsgnar]   55 (float6

In [ ]:
# ── SSBroyden: the merged Crunch, on the SAME objective and the SAME start ────────────
# Two runs: from the initial parameters (the far start) and polishing DSGNAR's result.
# Graded every iteration against the reference, not only by the loss.
def run_ssbroyden(x_start, n_iter, tag):
    x, H, hist = jnp.asarray(x_start), None, []
    t0 = time.perf_counter()
    for k in range(n_iter):
        opts = {"maxiter": 1, "gtol": 1e-9, "update_method": "ssbroyden2",
                "line_search": "wolfe", "max_iter": 100, "scale_invariant": True,
                "alphamin": 1e-15, "initial_scale": (k == 0)}
        if H is not None:
            opts["initial_H"] = H
        r = minimize_qn(paper_loss_qn, paper_loss_qn_grad, x, opts)
        x, H = r.x, r.hess_inv
        loss_now = float(r.fun)
        hist.append((k + 1, loss_now, rel_l2(x)["mean"]))
        if (k + 1) % 20 == 0 or k == 0:
            print(f"[ssb:{tag}] {k + 1:4d}  loss {loss_now:.4e}  relL2 {hist[-1][2]:.4f}")
    print(f"[ssb:{tag}] {n_iter} iterations in {time.perf_counter() - t0:.1f}s | "
          f"loss {float(paper_loss_qn(x_start)):.4e} -> {hist[-1][1]:.4e} | "
          f"rel L2 {hist[-1][2]:.4f}")
    return x, hist

qn_flat, qn_hist = run_ssbroyden(flat0, QN_MAXITER, "from x0")
qn_pol_flat, qn_pol_hist = run_ssbroyden(ds_flat, QN_MAXITER, "polish")

# grade the candidates and keep the best by ACCURACY (the loss is not the metric)
cands = {"DSGNAR": ds_flat, "SSBroyden from x0": qn_flat, "SSBroyden polish": qn_pol_flat}
grades = {k: rel_l2(v)["mean"] for k, v in cands.items()}
best_name = min(grades, key=grades.get)
best = cands[best_name]
print("\nrel L2 (mean over channels) and paper loss:")
for k, v in cands.items():
    print(f"  {k:20s} relL2 {grades[k]:.4f}   loss {float(paper_loss_jit(v)):.4e}")
print(f"best by accuracy: {best_name}")

[ssb:from x0]    1  loss 1.5466e-01  relL2 3.4214
[ssb:from x0]   20  loss 1.3281e-02  relL2 1.0981
[ssb:from x0]   40  loss 4.3144e-03  relL2 0.7667
[ssb:from x0]   60  loss 1.2036e-03  relL2 0.6113
[ssb:from x0]   80  loss 6.6897e-04  relL2 0.6259
[ssb:from x0]  100  loss 4.5812e-04  relL2 0.6450
[ssb:from x0]  120  loss 3.6896e-04  relL2 0.6608
[ssb:from x0]  140  loss 3.2231e-04  relL2 0.6215
[ssb:from x0]  160  loss 2.8876e-04  relL2 0.6106
[ssb:from x0]  180  loss 2.6132e-04  relL2 0.5883
[ssb:from x0]  200  loss 2.2038e-04  relL2 0.5127
[ssb:from x0]  220  loss 1.7917e-04  relL2 0.3828
[ssb:from x0]  240  loss 1.3299e-04  relL2 0.3447
[ssb:from x0]  260  loss 1.0565e-04  relL2 0.2971
[ssb:from x0]  280  loss 9.0343e-05  relL2 0.3405
[ssb:from x0]  300  loss 7.9906e-05  relL2 0.3234
[ssb:from x0]  320  loss 7.1051e-05  relL2 0.2627
[ssb:from x0]  340  loss 6.0270e-05  relL2 0.2076
[ssb:from x0]  360  loss 5.1870e-05  relL2 0.1593
[ssb:from x0]  380  loss 4.4691e-05  relL2 0.1250


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.2))
it = [h[0] for h in ds_hist]
axs[0].semilogy(it, [h[1] for h in ds_hist], "-", lw=1.8, label="DSGNAR (paper code)")
axs[0].semilogy([h[0] for h in qn_hist], [h[1] for h in qn_hist], "-", lw=1.8,
                label="SSBroyden from x0")
axs[0].semilogy([h[0] for h in qn_pol_hist], [h[1] for h in qn_pol_hist], "--", lw=1.6,
                label="SSBroyden polish from DSGNAR")
axs[0].set(xlabel="iteration", ylabel="paper loss  $\\sum_i w_i\\,\\mathrm{mean}(r_i^2)$",
           title="loss")
axs[1].semilogy(it, [h[2] for h in ds_hist], "-", lw=1.8, label="DSGNAR (paper code)")
axs[1].semilogy([h[0] for h in qn_hist], [h[2] for h in qn_hist], "-", lw=1.8,
                label="SSBroyden from x0")
axs[1].semilogy([h[0] for h in qn_pol_hist], [h[2] for h in qn_pol_hist], "--", lw=1.6,
                label="SSBroyden polish from DSGNAR")
axs[1].set(xlabel="iteration", ylabel="rel $\\ell_2$ error vs reference", title="accuracy")
axs[1].axhline(grades["DSGNAR"], color="C0", ls=":", lw=1)
for a in axs:
    a.grid(alpha=0.3, which="both"); a.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGDIR / "optimizers_loss_and_relL2.png", dpi=140, bbox_inches="tight")
print("saved", FIGDIR / "optimizers_loss_and_relL2.png")
plt.show()

In [ ]:
# ── fields of the best-by-accuracy result ────────────────────────────────────────────
best_params = eqx.combine(unflatten(best), static)
x0, x1 = ref.plot_x0, ref.plot_x1
X, Y = np.meshgrid(x0, x1, indexing="ij")
grid_pts = np.stack([X.ravel(), Y.ravel()], 1)
pred_grid = np.asarray(_predict(best_params, jnp.asarray(grid_pts))).reshape(
    x0.size, x1.size, -1)
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for i, ch in enumerate(CHANNELS):
    p = pred_grid[..., i]
    e = np.array(ref.plot_grids[ch])
    if ch == "p":
        p, e = p - p.mean(), e - e.mean()
    err = np.abs(p - e)
    rl = np.linalg.norm(p - e) / np.linalg.norm(e)
    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [p, e, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rl:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"]):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel="$x$", ylabel="$y$", title=title)
        fig.colorbar(im, ax=ax)
    fig.savefig(FIGDIR / f"fields_{ch}.png", dpi=140, bbox_inches="tight")
    print("saved", FIGDIR / f"fields_{ch}.png")
    plt.show()

print(f"final: {best_name} | rel L2 " +
      ", ".join(f"{c} {v:.3f}" for c, v in rel_l2(best).items() if c in CHANNELS))